# CFP_Parametric_Benchmarks

Four parametric VaR benchmarks.

**Paper:** Pele, Lessmann, Hardle (2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from tqdm import tqdm
from arch import arch_model

DATA_DIR = Path('../cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'
OUT_DIR = DATA_DIR / 'benchmarks'
OUT_DIR.mkdir(exist_ok=True)

ASSETS = [
    'SP500','STOXX','GDAXI','CACT','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
WINDOW = 250

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

In [ ]:
# GJR-GARCH and GARCH-N
for asset in tqdm(ASSETS, desc='GARCH models'):
    ret = load_returns(asset)
    n = len(ret)
    for method, vol, dist in [('gjr_garch', 'GARCH', 'skewt'), ('garch_n', 'GARCH', 'normal')]:
        records = []
        for t in range(WINDOW, n):
            r_win = ret.iloc[t-WINDOW:t] * 100  # arch expects percentage
            try:
                if method == 'gjr_garch':
                    am = arch_model(r_win, vol='GARCH', p=1, o=1, q=1, dist=dist)
                else:
                    am = arch_model(r_win, vol='GARCH', p=1, q=1, dist=dist)
                res = am.fit(disp='off', show_warning=False)
                fc = res.forecast(horizon=1)
                mu = fc.mean.iloc[-1, 0] / 100
                sigma = np.sqrt(fc.variance.iloc[-1, 0]) / 100
            except Exception:
                mu, sigma = 0, r_win.std() / 100
            row = {'date': ret.index[t]}
            for alpha in ALPHAS:
                row[f'VaR_{alpha}'] = mu + sigma * stats.norm.ppf(alpha)
            row['mean'] = mu
            row['std'] = sigma
            records.append(row)
        df_out = pd.DataFrame(records).set_index('date')
        df_out.to_parquet(OUT_DIR / f'{asset}_{method}.parquet')

In [ ]:
# Historical Simulation and EWMA
LAMBDA = 0.94

for asset in tqdm(ASSETS, desc='HS + EWMA'):
    ret = load_returns(asset)
    n = len(ret)
    for method in ['hs', 'ewma']:
        records = []
        for t in range(WINDOW, n):
            r_win = ret.iloc[t-WINDOW:t].values
            row = {'date': ret.index[t]}
            if method == 'hs':
                for alpha in ALPHAS:
                    row[f'VaR_{alpha}'] = np.percentile(r_win, alpha * 100)
                row['mean'] = r_win.mean()
                row['std'] = r_win.std()
            else:  # EWMA
                weights = np.array([(1-LAMBDA)*LAMBDA**i for i in range(WINDOW-1, -1, -1)])
                weights /= weights.sum()
                sigma = np.sqrt(np.sum(weights * r_win**2))
                for alpha in ALPHAS:
                    row[f'VaR_{alpha}'] = sigma * stats.norm.ppf(alpha)
                row['mean'] = 0.0
                row['std'] = sigma
            records.append(row)
        df_out = pd.DataFrame(records).set_index('date')
        df_out.to_parquet(OUT_DIR / f'{asset}_{method}.parquet')

print(f'Done: {len(list(OUT_DIR.glob("*.parquet")))} parquet files')